# 准备数据

In [1]:
# 过滤Alphalens的warning
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# 加载模块
import polars as pl

from vnpy.trader.constant import Interval

from vnpy.alpha import AlphaLab

In [3]:
# 创建数据中心
lab: AlphaLab = AlphaLab("./lab/csi300")

In [4]:
# 设置任务参数
name = "300_lasso"
index_symbol: str = "000300.SSE"
start: str = "2008-01-01"
end: str = "2023-12-31"
interval: Interval = Interval.DAILY
extended_days: int = 100

In [5]:
# 加载所有成分股代码
component_symbols: list[str] = lab.load_component_symbols(index_symbol, start, end)

# 特征计算

In [6]:
# 加载模块
from functools import partial

from vnpy.trader.constant import Interval

from vnpy.alpha.dataset import (
    AlphaDataset,
    process_drop_na,
    process_cs_norm,
    process_fill_na
)
from vnpy.alpha.dataset.datasets.alpha_158 import Alpha158

In [7]:
# 加载成分股数据
df: pl.DataFrame = lab.load_bar_df(component_symbols, interval, start, end, extended_days)

In [8]:
df

datetime,open,high,low,close,volume,turnover,open_interest,vwap,vt_symbol
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,str
2007-09-24 00:00:00,0.964547,1.01545,0.946327,1.0,2.1114e7,2.25337684e8,0.0,10.67259,"""600583.SSE"""
2007-09-25 00:00:00,1.009911,1.016048,0.969896,0.975843,9.3685e6,1.01628813e8,0.0,10.847906,"""600583.SSE"""
2007-09-26 00:00:00,0.972476,0.987926,0.948708,0.951079,1.3598e7,1.44069994e8,0.0,10.595273,"""600583.SSE"""
2007-09-27 00:00:00,0.956627,0.969896,0.940788,0.954256,1.1484e7,1.20540173e8,0.0,10.496695,"""600583.SSE"""
2007-09-28 00:00:00,0.964547,0.993266,0.950291,0.965343,2.5747e7,2.74751943e8,0.0,10.671334,"""600583.SSE"""
…,…,…,…,…,…,…,…,…,…
2024-01-04 00:00:00,0.24143,0.242044,0.238973,0.240201,9.442352e6,3.6975e7,0.0,3.915856,"""002570.SZSE"""
2024-01-05 00:00:00,0.240201,0.24143,0.235287,0.236516,1.1143e7,4.3208271e7,0.0,3.877616,"""002570.SZSE"""
2024-01-08 00:00:00,0.236516,0.236516,0.230987,0.230987,1.63885e7,6.2241837e7,0.0,3.797897,"""002570.SZSE"""


In [9]:
# 创建数据集对象
dataset: AlphaDataset = Alpha158(
    df,
    train_period = ("2008-01-01", "2014-12-31"),
    valid_period = ("2015-01-01", "2016-12-31"),
    test_period = ("2017-01-01", "2020-8-31"),
)

In [10]:
# 添加数据预处理器
dataset.add_processor("learn", partial(process_drop_na, names=["label"]))
dataset.add_processor("learn", partial(process_cs_norm, names=["label"], method="zscore"))

dataset.add_processor("infer", partial(process_fill_na, fill_value=0))

In [11]:
# 收集指数成分过滤器
filters: dict[str, list[str]] = lab.load_component_filters(index_symbol, start, end)

In [12]:
# 准备特征和标签数据
dataset.prepare_data(filters, max_workers=3)

2026-10-06 07:45:52 开始计算表达式因子特征


100%|██████████| 159/159 [04:50<00:00,  1.82s/it]


2026-10-06 07:50:45 开始合并结果数据因子特征


0it [00:00, ?it/s]


2026-10-06 07:50:46 开始筛选成分股数据


100%|██████████| 819/819 [00:07<00:00, 113.33it/s]


In [13]:
# 数据预处理
dataset.process_data()

In [14]:
# 特征表现分析
dataset.show_feature_performance("rsv_5")

In [15]:
# 保存到文件缓存
lab.save_dataset(name, dataset)

# 模型训练

In [16]:
# 加载模块
import numpy as np

from vnpy.alpha import Segment, AlphaDataset, AlphaModel

from vnpy.alpha.model.models.lasso_model import LassoModel

In [17]:
# 从文件缓存加载
dataset: AlphaDataset = lab.load_dataset(name)

In [18]:
# 创建模型对象
model: AlphaModel = LassoModel()

In [19]:
# 使用数据集训练模型
model.fit(dataset)

d:\veighna_studio\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning:

Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.265e+05, tolerance: 6.548e+01



In [20]:
# 查看模型细节
model.detail()

2026-10-06 07:54:57 LASSO模型特征总数量: 46
2026-10-06 07:54:57 roc_30: 0.201147
2026-10-06 07:54:57 ma_5: -0.135721
2026-10-06 07:54:57 rsv_30: 0.093490
2026-10-06 07:54:57 klow_2: -0.066172
2026-10-06 07:54:57 rank_10: -0.063976
2026-10-06 07:54:57 rank_5: -0.051165
2026-10-06 07:54:57 rsv_20: 0.047866
2026-10-06 07:54:57 rank_60: 0.041962
2026-10-06 07:54:57 qtld_5: -0.039833
2026-10-06 07:54:57 max_60: 0.038590
2026-10-06 07:54:57 imxd_5: -0.033326
2026-10-06 07:54:57 rsqr_5: -0.031732
2026-10-06 07:54:57 cord_20: -0.031570
2026-10-06 07:54:57 rsqr_10: -0.026283
2026-10-06 07:54:57 imax_30: -0.025676
2026-10-06 07:54:57 sumd_30: 0.024993
2026-10-06 07:54:57 ksft_2: -0.019634
2026-10-06 07:54:57 rsv_10: 0.018945
2026-10-06 07:54:57 cord_30: -0.017276
2026-10-06 07:54:57 cord_10: -0.014624
2026-10-06 07:54:57 cord_5: -0.013932
2026-10-06 07:54:57 wvma_30: -0.012675
2026-10-06 07:54:57 rsqr_30: -0.011181
2026-10-06 07:54:57 imin_10: 0.011088
2026-10-06 07:54:57 sumd_10: 0.010947
2026-10-06 0

In [21]:
# 保存模型
lab.save_model(name, model)

# 预测信号

In [22]:
model: AlphaModel = lab.load_model(name)

In [23]:
# 用模型在测试集上预测
pre: np.ndarray = model.predict(dataset, Segment.TEST)

# 加载测试集数据
df_t: pl.DataFrame = dataset.fetch_infer(Segment.TEST)

# 合并预测信号列
df_t = df_t.with_columns(pl.Series(pre).alias("signal"))

# 提取信号数据
signal: pl.DataFrame = df_t["datetime", "vt_symbol", "signal"]

In [24]:
# 检查信号绩效
dataset.show_signal_performance(signal)

In [25]:
# 保存信号数据
lab.save_signal(name, signal)

# 策略回测

In [26]:
# 加载模块
import importlib
from datetime import datetime

from vnpy.alpha.strategy import BacktestingEngine

import vnpy.alpha.strategy.strategies.equity_demo_strategy as equity_demo_strategy

In [27]:
# 重载策略类
importlib.reload(equity_demo_strategy)
EquityDemoStrategy = equity_demo_strategy.EquityDemoStrategy

In [28]:
# 从文件加载信号数据
signal = lab.load_signal(name)

In [29]:
# 创建回测引擎对象
engine = BacktestingEngine(lab)

# 设置回测参数
engine.set_parameters(
    vt_symbols=component_symbols,
    interval=Interval.DAILY,
    start=datetime(2017, 1, 1),
    end=datetime(2020, 8, 1),
    capital=100000000
)

# 添加策略实例
setting = {"top_k": 30, "n_drop": 3, "hold_thresh": 3}
engine.add_strategy(EquityDemoStrategy, setting, signal)

In [30]:
# 执行回测任务
engine.load_data()
engine.run_backtesting()
engine.calculate_result()
engine.calculate_statistics()
engine.show_chart()

2026-10-06 07:55:08 开始加载历史数据


100%|██████████| 819/819 [00:08<00:00, 91.74it/s] 

2026-10-06 07:55:17 部分合约历史数据为空：['600357.SSE', '300866.SZSE', '300919.SZSE', '301269.SZSE', '688271.SSE', '601299.SSE', '601825.SSE', '605117.SSE', '688187.SSE', '600786.SSE', '601995.SSE', '688041.SSE', '600087.SSE', '600832.SSE', '300957.SZSE', '300888.SZSE', '601268.SSE', '601868.SSE', '688065.SSE', '600001.SSE', '000024.SZSE', '601059.SSE', '600631.SSE', '600941.SSE', '600938.SSE', '688303.SSE', '001289.SZSE', '300896.SZSE', '000562.SZSE', '600591.SSE', '000527.SZSE', '605499.SSE', '600102.SSE', '300999.SZSE', '600905.SSE', '601728.SSE', '300979.SZSE', '688223.SSE']
2026-10-06 07:55:17 所有历史数据加载完成
2026-10-06 07:55:17 策略初始化完成
2026-10-06 07:55:17 开始回放历史数据


2026-10-06 07:55:23 历史数据回放结束
2026-10-06 07:55:23 开始计算逐日盯市盈亏
2026-10-06 07:55:24 逐日盯市盈亏计算完成
2026-10-06 07:55:24 开始计算策略统计指标
2026-10-06 07:55:24 ------------------------------
2026-10-06 07:55:24 首个交易日：  2017-01-03
2026-10-06 07:55:24 最后交易日：  2020-07-31
2026-10-06 07:55:24 总交易日：  871
2026-10-06 07:55:24 盈利交易日：  467
2026-10-06 07:55:24 亏损交易日：  403
2026-10-06 07:55:24 起始资金：  100,000,000.00
2026-10-06 07:55:24 结束资金：  231,711,555.63
2026-10-06 07:55:24 总收益率：  131.71%
2026-10-06 07:55:24 年化收益：  36.29%
2026-10-06 07:55:24 最大回撤:   -42,438,182.84
2026-10-06 07:55:24 百分比最大回撤: -27.33%
2026-10-06 07:55:24 最长回撤天数:   150
2026-10-06 07:55:24 总盈亏：  131,711,555.63
2026-10-06 07:55:24 总手续费：  16,816,267.15
2026-10-06 07:55:24 总成交金额：  22,449,155,430.72
2026-10-06 07:55:24 总成交笔数：  4860
2026-10-06 07:55:24 日均盈亏：  151,218.78
2026-10-06 07:55:24 日均手续费：  19,306.85
2026-10-06 07:55:24 日均成交金额：  25,774,001.64
2026-10-06 07:55:24 日均成交笔数：  5.579793340987371
2026-10-06 07:55:24 日均收益率：  0.11%
2026-10-06 07:55:24 收益标准差：

In [31]:
# 显示超额收益分析结果
engine.show_performance(benchmark_symbol=index_symbol)